# 01 — Getting started with `LammpsClient`

This notebook walks through the lifecycle of a LAMMPS session: create → start
→ run → inspect → dispose. Run the cells in order; state lives on
`globalThis` so each cell can build on the previous one.

In [ ]:
// Load lammps.js (served by this site under ./lammps/). Run this cell first.
// The site root is derived from wherever this code runs: the kernel iframe
// inherits the page URL ({site}/lab/…), the worker kernel lives under
// {site}/extensions/….
const base = globalThis.document?.baseURI ?? location.href;
globalThis.SITE ??= base.replace(/(extensions|lab|notebooks|files|tree|repl|consoles|edit)\/.*$/, "");
globalThis.LammpsClient ??= (await import(new URL("lammps/client.js", globalThis.SITE))).LammpsClient;
"lammps.js loaded ✓"

## Create and start a session

`LammpsClient.create()` loads the wasm module. `print`/`printErr` receive
LAMMPS's stdout/stderr line by line. `start()` boots the LAMMPS instance
(equivalent to launching the `lmp` binary).

In [ ]:
globalThis.lammps = await LammpsClient.create({
  print: (line) => console.log(line),
  printErr: (line) => console.warn(line),
});
lammps.start();
console.log("session ready:", lammps.instance.isReady());

## Build a system with `runScript`

`runScript` executes a whole LAMMPS input at once. Any valid LAMMPS input
works — this creates a small Lennard-Jones crystal.

In [ ]:
lammps.runScript(`
  units         lj
  atom_style    atomic
  lattice       fcc 0.8442
  region        box block 0 3 0 3 0 3
  create_box    1 box
  create_atoms  1 box
  mass          1 1.0
  velocity      all create 3.0 87287
  pair_style    lj/cut 2.5
  pair_coeff    1 1 1.0 1.0 2.5
  fix           1 all nve
  thermo        100
`);
console.log("atoms created:", lammps.syncParticles().count);

## Single commands with `runCommand`

`runCommand` runs one LAMMPS command — handy for interactive work. The session
keeps all its state between cells, so you can run a bit, look at the system,
and run some more.

In [ ]:
lammps.runCommand("run 300");
console.log("current step:", lammps.getCurrentStep());

In [ ]:
// Run some more — the simulation continues where it left off.
lammps.runCommand("run 300");
console.log("current step:", lammps.getCurrentStep());

## Clean up

`dispose()` stops the instance so the wasm module can be garbage collected.
(Re-run the *create* cell above to start fresh — each `create()` call gives an
independent LAMMPS session.)

In [ ]:
lammps.dispose();
delete globalThis.lammps;
"done"